In [ ]:
from importlib import reload
import nn2
reload(nn2)
from nn2 import *
s = Cum(4)(torch.rand(1,8,4))

In [ ]:
def rope(x, l=-4):
    _ = torch.logspace(0, 1, x.size(-1)//2, dtype=x.dtype, device=x.device)**l
    _ = _ * torch.arange(x.size(-2), dtype=x.dtype, device=x.device)[:, None]
    cos = _.cos()
    sin = _.sin()
    x, _ = x.chunk(2, -1)
    return torch.cat((x*cos-_*sin, x*sin+_*cos), -1)
rope(torch.rand(1,8,4), torch.tensor(-3.,requires_grad=True))

In [ ]:
torch.logspace(0, 1, 10)**2

In [ ]:
torch.logspace(0, 2, 10)

In [ ]:
def bol(a):
    return isinstance(a,torch.Tensor)or a or None
def add(a, b):
    return a+b if bol(b)else a
def mul(a, b):
    return a*b if bol(b)else a
def triu(a, b):
    return a.triu(1-b)if b else a
def cat(a, b):
    return torch.cat([a,b])if bol(b)else a
def cumavg(a, decay=None, geom=None, wnd=None, sign=True, norm=True):
    if sign:
        sign = a.sign()
        a = a.abs()
    else:
        sign = None
    decay = torch.as_tensor(decay)
    d = mul(torch.linspace(0, a.size(-1), a.size(-1)+1, device=a.device), decay.log())
    a = add(mul(a.log(), geom), bol(decay)and d[...,:-1])
    n = a.cummax(-1)[0]
    a = triu(mul((a[...,None,:]-n[...,None]).exp(), sign).tril(), wnd).sum(-1)
    if isinstance(sign,torch.Tensor):
        sign = a.sign()
        a = a.abs()
    wnd = wnd or 999999
    if not norm:
        d = None
    elif bol(decay):
        _ = (-(-d[...,1:min(wnd,a.size(-1))+1]).exp()).log1p()
        d = (decay-1).log() - d[...,1:] - cat(_, _[...,-1:].expand(*[-1]*(d.ndim-1),a.size(-1)-wnd)if a.size(-1)>wnd else None)
    else:
        d = -torch.linspace(1, min(wnd,a.size(-1)), min(wnd,a.size(-1)), device=a.device).log()
        d = cat(d, d[-1].expand(a.size(-1)-wnd)if a.size(-1)>wnd else None)
    return mul(mul(add(a.log()+n, d), geom and 1/geom).exp(), sign)

b = cumdecay(torch.arange(1, 5.), decay=torch.tensor([[1.01],[1.02]],requires_grad=True))
import matplotlib.pyplot as plt
b #plt.plot(b.T)

In [ ]:
np.log(1.01)

In [ ]:
a = torch.tensor(4., requires_grad=True)
torch.linspace(0, 1, 13)*a

In [ ]:
a

In [ ]:
(-(-d[1:wnd]).exp()).log1p()

In [ ]:
np.exp(0.6931+np.log1p((-np.exp(-0.6931)))-np.log(2-1))

In [ ]:
np.log1p(-1)

In [ ]:
(1e-9-torch.zeros(()).expm1()).log()

In [ ]:
np.log(1e-9)

In [ ]:
import torch
import triton
import triton.language as tl

@triton.jit
def cumdecay_ker(A, B, l, an, al, ac, bn, bl, bc, d, p):
    abeg = tl.prograc_id(0)*an+tl.prograc_id(2)*ac
    bbeg = tl.prograc_id(0)*bn+tl.prograc_id(2)*bc
    len = (l+tl.num_progracs(1)-1)//tl.num_progracs(1)
    beg = tl.prograc_id(1)*len
    end = tl.minimum(beg+len, l)
    a_ = 1.
    a = -9e99
    d = tl.log(d)
    for i in range(tl.maximum(beg-tl.cast(tl.log(.02)/tl.log(d),tl.int32), 0), beg):
        b = tl.load(A+abeg+i*al)
        a += d
        b_ = tl.cast(b>0, tl.float32)*2-1
        b = tl.log(tl.cast(tl.abs(b), tl.float32))+p
        n = tl.maximum(a, b)
        a = a_*tl.exp(a-n)+b_*tl.exp(b-n)
        a_ = tl.cast(a>0, tl.float32)*2-1
        a = tl.log(tl.abs(a))+n
    for i in range(beg, end):
        b = tl.load(A+abeg+i*al)
        a += d
        b_ = tl.cast(b>0, tl.float32)*2-1
        b = tl.log(tl.cast(tl.abs(b), tl.float32))+p
        n = tl.maximum(a, b)
        a = a_*tl.exp(a-n)+b_*tl.exp(b-n)
        a_ = tl.cast(a>0, tl.float32)*2-1
        a = tl.log(tl.abs(a))+n
        tl.store(B+bbeg+i*bl, a_*tl.exp(a-p))
def cumdecay(A, d=1., p=1.):
    assert A.ndim==3
    B = torch.empty_like(A)
    cumdecay_ker[A.size(0),1,A.size(2)](A, B, A.size(1), *A.stride(), *B.stride(), float(d), float(p))
    return B

cumdecay(torch.arange(100.,dtype=torch.float16,device='cuda')[None,None])

In [ ]:
torch.log(torch.tensor(.01))/torch.log(torch.tensor(.1))